# Retrieving Zernike wavefront modes from Pyramid WFS images with a CNN

**OAO School 2026 — Machine Learning for Adaptive Optics, hands-on tutorial**

---

### What is the problem?

A Pyramid Wavefront Sensor (PWFS) produces a **4-channel image**. Here the machine-learning task is

\[
\boxed{\text{PWFS image }(4,H,W)\;\longrightarrow\;\text{first }N_Z\leq 50\text{ Zernike coefficients}}
\]

This is a **multi-output regression** problem. Because the input is an image, we use a **convolutional neural network (CNN)**.

The tutorial keeps the same pedagogical structure as the original scalar-input tutorial:

1. physics and data inspection,
2. train / validation / test split,
3. train-only normalization,
4. PyTorch `Dataset` / `DataLoader`,
5. CNN architecture,
6. training and validation,
7. test-set diagnostics,
8. residual analysis,
9. saving and reusing the model,
10. exercises.

## 0. Setup

We import the libraries used throughout the notebook and fix random seeds.

The **dataset-loading cell is intentionally isolated**. The rest of the notebook expects only:

- `X.shape = (N, 4, H, W)` — four-channel PWFS images
- `y.shape = (N, N_Z)` — Zernike coefficients

where `N_Z` is at most 50.

In [ ]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split

ROOT = Path.cwd()

# EDIT THIS if your data are somewhere else.
DATA_ROOT = ROOT / "pyramid_wfs_dataset"

OUTPUT_ROOT = ROOT / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("Data root:", DATA_ROOT)
print("PyTorch :", torch.__version__)
print("Device  :", device)
if device.type == "cuda":
    print("GPU     :", torch.cuda.get_device_name(0))

In [ ]:
plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "lines.linewidth": 2,
})

## 1. The physics: PWFS images → Zernike modes

A wavefront can be expanded as

\[
W(\rho,\theta)=\sum_j a_j Z_j(\rho,\theta),
\]

where \(a_j\) is the coefficient of one Zernike mode.

The CNN learns the inverse mapping

\[
\mathrm{PWFS}(x,y,c)\rightarrow
(a_0,a_1,\ldots,a_{N_Z-1}).
\]

**The Zernike convention matters.** The ordering, normalization, and whether piston is included must match the labels used to generate the dataset.

### Why a CNN?

A dense network would flatten the four images into one very long vector and discard their natural spatial structure. A CNN instead learns local spatial patterns and progressively combines them into larger features:

```text
4 PWFS channels
      │
 Conv + activation
      │
 downsampling
      │
 deeper convolutions
      │
 compact feature representation
      │
 fully-connected layers
      │
 N_Z Zernike coefficients
```

The final layer is linear because this is regression and Zernike coefficients can be positive or negative.

## 2. Load the dataset and look at it

### 2.1 Expected data format

The loader below supports either:

```text
pyramid_wfs_dataset/
    X.npy
    y.npy
    zernike_names.npy       # optional
```

or:

```text
pyramid_wfs_dataset/
    dataset.npz             # arrays X, y, optionally zernike_names
```

If your PWFS data are in FITS files, replace only the loader cell. Everything after it uses the same `X` and `y` interface.

If more than 50 modes are available, the notebook selects the first 50 automatically. If fewer are available, it uses all of them.

In [ ]:
# Supported formats:
#   1) dataset.npz with arrays X, y, optionally zernike_names
#   2) X.npy + y.npy, optionally zernike_names.npy
#
# Required shapes:
#   X = (N, 4, H, W)
#   y = (N, N_Z)

NPZ_PATH = DATA_ROOT / "dataset.npz"
X_PATH = DATA_ROOT / "X.npy"
Y_PATH = DATA_ROOT / "y.npy"
ZERN_NAMES_PATH = DATA_ROOT / "zernike_names.npy"

if NPZ_PATH.exists():
    data = np.load(NPZ_PATH, allow_pickle=True)
    X_all = data["X"]
    y_all = data["y"]
    zernike_names_all = (
        data["zernike_names"] if "zernike_names" in data.files else None
    )
elif X_PATH.exists() and Y_PATH.exists():
    X_all = np.load(X_PATH)
    y_all = np.load(Y_PATH)
    zernike_names_all = (
        np.load(ZERN_NAMES_PATH, allow_pickle=True)
        if ZERN_NAMES_PATH.exists() else None
    )
else:
    raise FileNotFoundError(
        f"Could not find {NPZ_PATH} or {X_PATH} + {Y_PATH}. "
        "Adapt this cell to your PWFS dataset."
    )

X_all = np.asarray(X_all, dtype=np.float32)
y_all = np.asarray(y_all, dtype=np.float32)

if X_all.ndim != 4 or X_all.shape[1] != 4:
    raise ValueError(f"Expected X shape (N, 4, H, W), got {X_all.shape}")
if y_all.ndim != 2:
    raise ValueError(f"Expected y shape (N, N_Z), got {y_all.shape}")
if len(X_all) != len(y_all):
    raise ValueError("X and y contain different numbers of samples.")

N_ZERNIKE = min(50, y_all.shape[1])

X = X_all
y = y_all[:, :N_ZERNIKE]

if zernike_names_all is None:
    TARGET_NAMES = [f"Z{i+1}" for i in range(N_ZERNIKE)]
else:
    TARGET_NAMES = [str(v) for v in zernike_names_all[:N_ZERNIKE]]

print("PWFS input shape:", X.shape)
print("Target shape    :", y.shape)
print("Number of modes :", N_ZERNIKE)
print("Image size      :", X.shape[-2:])
print("First modes     :", TARGET_NAMES[:10])

### 2.2 Sanity checks

Before training anything, check:

- all four PWFS channels are present,
- all values are finite,
- image dimensions are identical,
- target coefficients are finite,
- target modes have sensible dynamic ranges.

A network can train on a broken dataset, so these checks are part of the tutorial rather than an afterthought.

In [ ]:
print("Finite WFS values   :", np.isfinite(X).all())
print("Finite Zernike vals :", np.isfinite(y).all())
print("WFS min / max       :", float(X.min()), float(X.max()))
print("WFS mean / std      :", float(X.mean()), float(X.std()))

summary = pd.DataFrame({
    "mean": y.mean(0),
    "std": y.std(0),
    "min": y.min(0),
    "max": y.max(0),
}, index=TARGET_NAMES)

display(summary.round(4))

### 2.3 Visualise the four Pyramid WFS channels

A single sample is much more informative than a table of numbers.

The same wavefront is represented in four channels, so we plot the four images side by side using a common intensity scale.

In [ ]:
k = 0

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2), constrained_layout=True)

vmin, vmax = X[k].min(), X[k].max()

for c, ax in enumerate(axes):
    im = ax.imshow(X[k, c], origin="lower", vmin=vmin, vmax=vmax)
    ax.set_title(f"PWFS channel {c+1}")
    ax.set_xlabel("x [pixel]")
    ax.set_ylabel("y [pixel]")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)

fig.suptitle(f"Example PWFS measurement — sample {k}")
plt.show()

### 2.4 Look at several random WFS samples

Do not inspect only one image. Random samples reveal whether the dataset contains real variation and whether there are pathological or empty measurements.

In [ ]:
rng = np.random.default_rng(SEED)
examples = rng.choice(len(X), size=min(4, len(X)), replace=False)

fig, axes = plt.subplots(
    len(examples), 4,
    figsize=(10, 2.3 * len(examples)),
    squeeze=False,
    constrained_layout=True,
)

for row, idx in enumerate(examples):
    for c in range(4):
        axes[row, c].imshow(X[idx, c], origin="lower")
        axes[row, c].set_title(f"sample {idx}, ch {c+1}")
        axes[row, c].axis("off")

plt.show()

### 2.5 Look at the Zernike target distribution

A mode that is almost never excited is difficult to learn reliably.

We inspect the first few coefficient distributions and then plot the standard deviation of all selected modes.

In [ ]:
n_show = min(12, N_ZERNIKE)

fig, axes = plt.subplots(3, 4, figsize=(12, 7))

for j, ax in enumerate(axes.flat):
    if j < n_show:
        ax.hist(y[:, j], bins=25, edgecolor="white", linewidth=1)
        ax.set_title(TARGET_NAMES[j])
        ax.set_xlabel("coefficient")
        ax.set_ylabel("count")
    else:
        ax.axis("off")

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(np.arange(N_ZERNIKE), y.std(0))
ax.set_xlabel("Zernike mode index")
ax.set_ylabel("target standard deviation")
ax.set_title("Dynamic range of the Zernike targets")
plt.show()

## 3. Split the data: train / validation / test

We divide the samples into three **disjoint** sets:

| Set | Used for | Does the model learn from it? |
|---|---|---|
| **Training** | update CNN weights | yes |
| **Validation** | choose the best epoch / hyper-parameters | no |
| **Test** | final unbiased evaluation | no |

### Important for AO simulations

If several images come from the **same simulation, atmospheric realization, or time sequence**, do not randomly split individual frames. Split by simulation/realization instead.

Otherwise almost identical images can occur in both training and test and produce an artificially good score.

The code below assumes each row is an independent sample.

In [ ]:
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

indices = np.arange(len(X))

train_idx, rest_idx = train_test_split(
    indices,
    test_size=VAL_FRAC + TEST_FRAC,
    random_state=SEED,
)

val_idx, test_idx = train_test_split(
    rest_idx,
    test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC),
    random_state=SEED,
)

assert not (set(train_idx) & set(val_idx))
assert not (set(train_idx) & set(test_idx))
assert not (set(val_idx) & set(test_idx))

print(f"train: {len(train_idx)}")
print(f"val  : {len(val_idx)}")
print(f"test : {len(test_idx)}")

### 3.1 Check the target ranges in the three sets

A test value far outside the training range is an extrapolation problem. Neural networks are generally much better at interpolation than extrapolation.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

for ax, idx, title in zip(
    axes,
    [train_idx, val_idx, test_idx],
    ["training", "validation", "test"],
):
    ax.boxplot(y[idx], showfliers=False)
    ax.set_title(title)
    ax.set_xlabel("Zernike mode")
    ax.set_ylabel("coefficient")
    ax.grid(axis="x", visible=False)

plt.tight_layout()
plt.show()

## 4. Normalize the data — using the training set only

We standardize the WFS images **channel by channel**:

\[
x'_{c,h,w}=(x_{c,h,w}-\mu_c)/\sigma_c.
\]

The means and standard deviations are computed from the **training images only**.

The Zernike coefficients are also standardized mode by mode:

\[
y'_j=(y_j-\mu_j)/\sigma_j.
\]

This prevents large-amplitude modes from dominating the MSE loss.

Using validation or test images to compute these statistics would be **data leakage**.

In [ ]:
# Start with False. If the PWFS intensities have a very large
# dynamic range, compare against USE_LOG1P=True as an exercise.
USE_LOG1P = False

X_work = (
    np.log1p(np.clip(X, 0, None))
    if USE_LOG1P else X.copy()
)

# Training-set statistics only.
x_mean = X_work[train_idx].mean(axis=(0, 2, 3), keepdims=True)
x_std  = X_work[train_idx].std(axis=(0, 2, 3), keepdims=True)

y_mean = y[train_idx].mean(axis=0)
y_std  = y[train_idx].std(axis=0)

x_std[x_std < 1e-8] = 1.0
y_std[y_std < 1e-8] = 1.0

Xn = (X_work - x_mean) / x_std
yn = (y - y_mean) / y_std

def denormalize_y(yn_):
    return yn_ * y_std + y_mean

print("Input means:", x_mean.reshape(4).round(4))
print("Input stds :", x_std.reshape(4).round(4))
print("Target std:", y_std.round(4))

## 5. PyTorch `Dataset` and `DataLoader`

PyTorch uses the image convention

\[
(N,C,H,W),
\]

which matches our PWFS tensor directly.

The training loader shuffles samples; validation and test loaders do not.

In [ ]:
BATCH_SIZE = 32

def make_loader(idx, shuffle):
    ds = TensorDataset(
        torch.tensor(Xn[idx], dtype=torch.float32),
        torch.tensor(yn[idx], dtype=torch.float32),
    )
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=(device.type == "cuda"),
    )

train_loader = make_loader(train_idx, shuffle=True)
val_loader   = make_loader(val_idx, shuffle=False)
test_loader  = make_loader(test_idx, shuffle=False)

xb, yb = next(iter(train_loader))

print("one batch -> inputs :", tuple(xb.shape))
print("one batch -> targets:", tuple(yb.shape))

## 6. The convolutional neural network

We use a compact CNN:

```text
input (4,H,W)
     │
 Conv 3×3, 32 + ReLU
     │
 MaxPool
     │
 Conv 3×3, 64 + ReLU
     │
 MaxPool
     │
 Conv 3×3, 128 + ReLU
     │
 Adaptive average pooling
     │
 Linear → 128
     │
 ReLU + Dropout
     │
 Linear → N_Z
     │
 Zernike coefficients
```

Early layers learn local PWFS structures; deeper layers combine them into larger spatial features.

The output layer has **no activation** because this is regression.

In [ ]:
N_INPUT_CHANNELS = 4
N_OUTPUTS = N_ZERNIKE

class PyramidWFS_CNN(nn.Module):
    def __init__(self, n_outputs, base_channels=32, dropout=0.15):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(N_INPUT_CHANNELS, base_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(base_channels),
            nn.MaxPool2d(2),

            nn.Conv2d(base_channels, 2 * base_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(2 * base_channels),
            nn.MaxPool2d(2),

            nn.Conv2d(2 * base_channels, 4 * base_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(4 * base_channels),

            # Gives a fixed-size feature map even if H,W change.
            nn.AdaptiveAvgPool2d((4, 4)),
        )

        n_features = 4 * base_channels * 4 * 4

        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(n_features, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, n_outputs),
        )

    def forward(self, x):
        return self.regressor(self.features(x))


torch.manual_seed(SEED)

model = PyramidWFS_CNN(
    n_outputs=N_OUTPUTS,
    base_channels=32,
    dropout=0.15,
).to(device)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-5,
)

print(model)
print(
    f"\nTrainable parameters: "
    f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}"
)

### 6.1 Check the tensor shapes

Always do one forward pass before starting training. This catches channel-order and tensor-shape mistakes immediately.

In [ ]:
with torch.no_grad():
    test_out = model(xb.to(device))

print("Input :", tuple(xb.shape))
print("Output:", tuple(test_out.shape))
print("Expected:", (len(xb), N_ZERNIKE))

assert test_out.shape[1] == N_ZERNIKE

## 7. Training and validation

Each epoch is one full pass over the training set.

1. **Training:** forward pass → loss → back-propagation → optimizer update.
2. **Validation:** evaluate the loss without changing weights.

We restore the epoch with the **lowest validation loss**. This is a simple and useful protection against over-fitting.

In [ ]:
EPOCHS = 100

def run_epoch(loader, train):
    model.train(train)

    total_loss = 0.0
    n_samples = 0

    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            pred = model(xb)
            loss = criterion(pred, yb)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * len(xb)
            n_samples += len(xb)

    return total_loss / n_samples


def save_checkpoint(path, history, best_epoch, best_val):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    torch.save({
        "model": {k: v.cpu() for k, v in model.state_dict().items()},
        "model_kwargs": {
            "n_outputs": N_OUTPUTS,
            "base_channels": 32,
            "dropout": 0.15,
        },
        "epoch": best_epoch,
        "best_val_loss": best_val,
        "history": history,
        "normalizer": {
            "use_log1p": USE_LOG1P,
            "input_mean": x_mean,
            "input_std": x_std,
            "output_mean": y_mean,
            "output_std": y_std,
        },
        "target_names": TARGET_NAMES,
        "input_shape": X.shape[1:],
        "split": {
            "seed": SEED,
            "train": train_idx,
            "val": val_idx,
            "test": test_idx,
        },
    }, path)

    print("Model saved to:", path)


def train_model(epochs=EPOCHS, save_path=None):
    history = {"train": [], "val": []}

    best_val = float("inf")
    best_epoch = -1
    best_state = None

    for epoch in range(epochs):
        train_loss = run_epoch(train_loader, True)
        val_loss = run_epoch(val_loader, False)

        history["train"].append(train_loss)
        history["val"].append(val_loss)

        if val_loss < best_val:
            best_val = val_loss
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

        if epoch % 10 == 0 or epoch == epochs - 1:
            print(
                f"epoch {epoch+1:3d}/{epochs} | "
                f"train loss {train_loss:.4f} | "
                f"val loss {val_loss:.4f}"
            )

    model.load_state_dict(best_state)

    print(
        f"\nBest validation loss {best_val:.4f} "
        f"at epoch {best_epoch + 1}."
    )

    if save_path is not None:
        save_checkpoint(save_path, history, best_epoch, best_val)

    return history, best_epoch, best_val

### 7.1 Train the CNN

Start with a modest number of epochs and inspect the learning curves. The correct epoch count depends strongly on dataset size and noise.

In [ ]:
CHECKPOINT_PATH = OUTPUT_ROOT / "pyramid_wfs_zernike_cnn.pt"

history, best_epoch, best_val = train_model(
    epochs=EPOCHS,
    save_path=CHECKPOINT_PATH,
)

print("Checkpoint:", CHECKPOINT_PATH)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))

ep = np.arange(1, len(history["train"]) + 1)

ax.plot(ep, history["train"], label="training")
ax.plot(ep, history["val"], label="validation")
ax.axhline(1.0, ls="--", lw=1.2, label="predict-the-mean baseline")
ax.axvline(best_epoch + 1, ls=":", lw=1.2)
ax.scatter([best_epoch + 1], [best_val], s=60, zorder=3)

ax.set_xlabel("epoch")
ax.set_ylabel("MSE loss (normalised Zernike coefficients)")
ax.set_title("CNN learning curves")
ax.legend()

plt.show()

### How to read the learning curves

- If training and validation both decrease, the CNN is learning a useful mapping.
- If training keeps decreasing while validation stops improving, the model is over-fitting.
- A validation loss below 1 is better than the simple baseline of predicting the training-set mean in normalized target space.
- Different Zernike modes can have very different difficulty. A good average loss does not mean every mode is equally well retrieved.

The best-validation checkpoint protects us from using the over-fitted final epoch.

## 8. Evaluation on the test set

The test set is now used for the first time.

For every Zernike mode we calculate:

- **MAE**
- **RMSE**
- **R²**
- **baseline MAE**

The important scientific question is:

> **Which Zernike modes can the PWFS + CNN retrieve reliably?**

In [ ]:
model.eval()

with torch.no_grad():
    pred_norm = model(
        torch.tensor(Xn[test_idx], dtype=torch.float32).to(device)
    )

pred_test = denormalize_y(pred_norm.cpu().numpy())
true_test = y[test_idx]

def r2_per_mode(true, pred):
    numerator = ((true - pred) ** 2).sum(axis=0)
    denominator = ((true - true.mean(axis=0)) ** 2).sum(axis=0)
    return np.where(
        denominator > 0,
        1 - numerator / denominator,
        np.nan,
    )

mae = np.abs(pred_test - true_test).mean(axis=0)
rmse = np.sqrt(((pred_test - true_test) ** 2).mean(axis=0))
mae_base = np.abs(y_mean - true_test).mean(axis=0)
r2_values = r2_per_mode(true_test, pred_test)

results = pd.DataFrame({
    "MAE (CNN)": mae,
    "RMSE (CNN)": rmse,
    "MAE (baseline)": mae_base,
    "improvement": 1 - mae / np.maximum(mae_base, 1e-12),
    "R²": r2_values,
}, index=TARGET_NAMES)

display(results.round(4))

### 8.1 Predicted versus true coefficients

The ideal result is a cloud around the diagonal:

\[
\hat a_j=a_j.
\]

A horizontal cloud means that the network has learned little about that mode and mostly predicts its mean.

In [ ]:
n_plot = min(N_ZERNIKE, 20)
ncols = 5
nrows = int(np.ceil(n_plot / ncols))

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(13, 2.6 * nrows),
    squeeze=False,
)

for j in range(n_plot):
    ax = axes.flat[j]

    lo = min(true_test[:, j].min(), pred_test[:, j].min())
    hi = max(true_test[:, j].max(), pred_test[:, j].max())
    pad = 0.05 * (hi - lo + 1e-12)

    ax.plot(
        [lo - pad, hi + pad],
        [lo - pad, hi + pad],
        ls="--",
        lw=1.2,
    )

    ax.scatter(
        true_test[:, j],
        pred_test[:, j],
        s=24,
        edgecolor="white",
        linewidth=0.7,
        zorder=3,
    )

    ax.set_xlim(lo - pad, hi + pad)
    ax.set_ylim(lo - pad, hi + pad)
    ax.set_aspect("equal")
    ax.set_title(f"{TARGET_NAMES[j]}  R²={r2_values[j]:+.2f}")
    ax.set_xlabel("true")
    ax.set_ylabel("predicted")

for ax in axes.flat[n_plot:]:
    ax.axis("off")

fig.suptitle(
    "Test set: predicted versus true Zernike coefficients",
    y=1.01,
)
plt.tight_layout()
plt.show()

### 8.2 Which modes are easiest?

Plotting the score against mode index makes it obvious where the retrieval becomes difficult.

For AO this can reveal a physical trend with spatial frequency, although the exact relationship depends on the Zernike ordering and on the PWFS/simulation setup.

In [ ]:
normalized_rmse = rmse / np.maximum(y.std(axis=0), 1e-12)

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(np.arange(1, N_ZERNIKE + 1), r2_values, marker="o", ms=3)
ax.axhline(0, ls="--", lw=1.2)
ax.set_xlabel("Zernike mode")
ax.set_ylabel("R²")
ax.set_title("Retrieval quality versus Zernike mode")
plt.show()

fig, ax = plt.subplots(figsize=(10, 3.8))
ax.plot(
    np.arange(1, N_ZERNIKE + 1),
    normalized_rmse,
    marker="o",
    ms=3,
)
ax.set_xlabel("Zernike mode")
ax.set_ylabel("RMSE / target std")
ax.set_title("Relative coefficient error versus Zernike mode")
plt.show()

### 8.3 Residuals

Define

\[
r_j=\hat a_j-a_j.
\]

Look for:

- non-zero mean → bias,
- broad distributions → large random error,
- errors growing with coefficient amplitude → possible non-linearity or saturation,
- a trend with mode number → some spatial frequencies are harder to retrieve.

In [ ]:
residuals = pred_test - true_test

n_plot = min(N_ZERNIKE, 12)

fig, axes = plt.subplots(3, 4, figsize=(12, 7))

for j, ax in enumerate(axes.flat):
    if j < n_plot:
        ax.hist(residuals[:, j], bins=15, edgecolor="white")
        ax.axvline(0, ls="--", lw=1.2)
        ax.set_title(
            f"{TARGET_NAMES[j]}\n"
            f"bias={residuals[:, j].mean():+.3g}"
        )
        ax.set_xlabel("prediction − truth")
        ax.set_ylabel("count")
    else:
        ax.axis("off")

plt.tight_layout()
plt.show()

### Interpretation

It is not necessary for all 50 modes to have high \(R^2\).

If low-order modes are retrieved well but high-order modes are not, possible causes include:

- PWFS spatial sampling,
- photon/read noise,
- insufficient diversity in the training set,
- the finite modal basis used in the simulation,
- a CNN that is too small,
- an unsuitable input representation,
- or genuinely weaker observability of those modes.

This is why the **per-mode diagnostics** are more informative than a single global MSE.

## 9. Saving and reusing the model

A useful checkpoint contains more than the weights. We also save:

- architecture parameters,
- WFS normalization statistics,
- Zernike normalization statistics,
- Zernike names/order,
- the split,
- training history,
- whether `log1p` was used.

Without the normalization statistics, inference on a new WFS image would not reproduce the preprocessing used during training.

In [ ]:
def load_checkpoint(path):
    path = Path(path)
    ckpt = torch.load(path, map_location=device, weights_only=False)

    loaded_model = PyramidWFS_CNN(**ckpt["model_kwargs"]).to(device)
    loaded_model.load_state_dict(ckpt["model"])
    loaded_model.eval()

    return loaded_model, ckpt


ckpt = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

for key, value in ckpt.items():
    if isinstance(value, dict):
        description = f"dict with keys {list(value.keys())}"
    elif isinstance(value, list):
        description = f"list with {len(value)} items"
    else:
        description = value
    print(f"{key:16s}: {description}")

### 9.1 Inference on a "new observation"

At inference time:

1. receive one `(4,H,W)` PWFS image,
2. apply the same optional transform,
3. normalize with the stored training statistics,
4. run the CNN,
5. undo target normalization.

The model does not need the true Zernike coefficients at inference time.

In [ ]:
retrieval_model, ckpt = load_checkpoint(CHECKPOINT_PATH)
norm = ckpt["normalizer"]

def retrieve_zernike(pwfs_4hw):
    # Predict Zernike coefficients from one (4,H,W) PWFS image.
    image = np.asarray(pwfs_4hw, dtype=np.float32)

    if image.shape != tuple(ckpt["input_shape"]):
        raise ValueError(
            f"Expected shape {tuple(ckpt['input_shape'])}, got {image.shape}"
        )

    if norm["use_log1p"]:
        image = np.log1p(np.clip(image, 0, None))

    image = (
        image - norm["input_mean"].squeeze(0)
    ) / norm["input_std"].squeeze(0)

    tensor = torch.tensor(
        image[None],
        dtype=torch.float32,
        device=device,
    )

    with torch.no_grad():
        prediction_norm = retrieval_model(tensor).cpu().numpy()[0]

    return (
        prediction_norm * norm["output_std"]
        + norm["output_mean"]
    )


# Demonstration: treat one test sample as a "new" observation.
k = test_idx[0]
estimate = retrieve_zernike(X[k])

comparison = pd.DataFrame({
    "true": y[k],
    "retrieved": estimate,
    "residual": estimate - y[k],
}, index=TARGET_NAMES)

display(comparison.round(5))

## 10. Exercises

1. **Hyper-parameters.** Change convolutional width, dropout, learning rate and batch size. Does validation improve?

2. **CNN depth.** Add a fourth convolutional block. Does it help, or does the larger model over-fit?

3. **Dense baseline.** Flatten the four images and train an MLP. Compare it with the CNN. Why is convolution a better inductive bias for images?

4. **Normalization.** Compare `USE_LOG1P=False` and `True`. Which gives better validation/test performance?

5. **Fewer outputs.** Train only the first 10 or 20 Zernike coefficients. Does reducing the number of outputs improve low-order retrieval?

6. **Noise robustness.** Add realistic photon/read noise. Train on a mixture of noise levels and test on a noise level not used during training.

7. **Generalisation.** Hold out an entire range of seeing or wavefront RMS instead of using a random split. How does extrapolation affect the result?

8. **Group-aware splitting.** If many frames come from the same atmospheric realization, split by realization ID. Compare with a naive random split.

9. **Mode ordering.** If your labels include radial and azimuthal orders, plot \(R^2\) against radial order rather than coefficient index.

10. **Input representation.** Compare raw four-channel images with flux-normalized images, channel differences, or another physically motivated PWFS representation.

11. **Spatial information.** Replace adaptive pooling with a larger flattened feature map. Does retaining more spatial information improve high-order modes?

12. **Uncertainty.** Train several CNNs with different seeds and use the spread of their predictions as a simple model-uncertainty estimate.

## 11. Final checklist

Before trusting the retrieval:

- [ ] Are the four PWFS channels in the correct order?
- [ ] Is the Zernike convention documented and consistent with the labels?
- [ ] Are the train/validation/test samples genuinely independent?
- [ ] Were normalization statistics computed from training data only?
- [ ] Does the CNN beat the predict-the-mean baseline?
- [ ] Which modes have useful \(R^2\)?
- [ ] Do the residuals show bias or systematic structure?
- [ ] Does the model survive realistic WFS noise?
- [ ] Does it generalize to simulations/observations outside the training distribution?

> **The CNN cannot recover information that is not present in the PWFS measurement or represented in the training data.**

Start from the physics, inspect the data, establish a baseline, and only then increase model complexity.